## Import necessary Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,confusion_matrix
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

## Import dataset

In [2]:
credit_data = pd.read_csv("credit_card_clean.csv")
credit_data

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,1,20000.0,female,university,married,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,female,university,single,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,female,university,single,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,female,university,married,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,male,university,married,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,29996,220000.0,male,highschool,married,39,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,29997,150000.0,male,highschool,single,43,-1,-1,-1,-1,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,29998,30000.0,male,university,single,37,4,3,2,-1,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,29999,80000.0,male,highschool,married,41,1,-1,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


## Data Understanding

In [3]:
credit_data.isna().sum()

ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3        0
PAY_4        0
PAY_5        0
PAY_6        0
BILL_AMT1    0
BILL_AMT2    0
BILL_AMT3    0
BILL_AMT4    0
BILL_AMT5    0
BILL_AMT6    0
PAY_AMT1     0
PAY_AMT2     0
PAY_AMT3     0
PAY_AMT4     0
PAY_AMT5     0
PAY_AMT6     0
DEFAULT      0
dtype: int64

In [4]:
credit_data.dtypes

ID             int64
LIMIT_BAL    float64
SEX           object
EDUCATION     object
MARRIAGE      object
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

In [5]:
credit_data.shape

(30000, 25)

## Data Preparation

In [6]:
del credit_data["ID"]

In [7]:
credit_data

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000.0,female,university,married,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,female,university,single,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,female,university,single,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,female,university,married,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,male,university,married,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,male,highschool,married,39,0,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,150000.0,male,highschool,single,43,-1,-1,-1,-1,0,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,30000.0,male,university,single,37,4,3,2,-1,0,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,80000.0,male,highschool,married,41,1,-1,0,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [8]:
le = LabelEncoder()
credit_data["SEX"] = le.fit_transform(credit_data["SEX"])
credit_data["EDUCATION"] = le.fit_transform(credit_data["EDUCATION"])
credit_data["MARRIAGE"] = le.fit_transform(credit_data["MARRIAGE"])
credit_data

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,20000.0,0,3,0,24,2,2,-1,-1,-2,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,120000.0,0,3,2,26,-1,2,0,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,90000.0,0,3,2,34,0,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,50000.0,0,3,0,37,0,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,50000.0,1,3,0,57,-1,0,-1,0,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,220000.0,1,1,0,39,0,0,0,0,0,...,88004.0,31237.0,15980.0,8500.0,20000.0,5003.0,3047.0,5000.0,1000.0,0
29996,150000.0,1,1,2,43,-1,-1,-1,-1,0,...,8979.0,5190.0,0.0,1837.0,3526.0,8998.0,129.0,0.0,0.0,0
29997,30000.0,1,3,2,37,4,3,2,-1,0,...,20878.0,20582.0,19357.0,0.0,0.0,22000.0,4200.0,2000.0,3100.0,1
29998,80000.0,1,1,0,41,1,-1,0,0,0,...,52774.0,11855.0,48944.0,85900.0,3409.0,1178.0,1926.0,52964.0,1804.0,1


In [9]:
credit_data.dtypes

LIMIT_BAL    float64
SEX            int64
EDUCATION      int64
MARRIAGE       int64
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

## Model Building

In [10]:
X = credit_data.drop("DEFAULT",axis=1)
y = credit_data["DEFAULT"]

In [11]:
X.shape,y.shape

((30000, 23), (30000,))

In [12]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=34,shuffle=True,stratify=y)

In [13]:
X_train.shape,y_train.shape

((24000, 23), (24000,))

In [14]:
X_test.shape,y_test.shape

((6000, 23), (6000,))

## Model Training

In [15]:
rf_model = RandomForestClassifier(max_depth=7)
rf_model.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,7
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## Model Testing

In [16]:
y_pred = rf_model.predict(X_test)

## Model Evaluation

In [17]:
print("Accuracy Score:",round(accuracy_score(y_test,y_pred),4))
print("Precision Score:",round(precision_score(y_test,y_pred),4))
print("Recall Score",round(recall_score(y_test,y_pred),4))
print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))

Accuracy Score: 0.815
Precision Score: 0.6782
Recall Score 0.3112
Confusion Matrix:
 [[4477  196]
 [ 914  413]]


## How to choose the Optimal Number of Features to train my Model?

In [18]:
def runRandomForestClassifier(X_train,X_test,y_train,y_test):
    rf_model = RandomForestClassifier(max_depth=7,random_state=123)
    rf_model.fit(X_train,y_train)
    
    y_pred = rf_model.predict(X_test)
    
    print("Accuracy Score:",round(accuracy_score(y_test,y_pred),4))
    print("Precision Score:",round(precision_score(y_test,y_pred),4))
    print("Recall Score",round(recall_score(y_test,y_pred),4))
    print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))

In [20]:
for i in range(1,25):
    rfe_model = RFE(estimator = GradientBoostingClassifier(),n_features_to_select=i)
    rfe_model.fit(X_train,y_train)
    X_train_rfe_model = rfe_model.transform(X_train)
    X_test_rfe_model = rfe_model.transform(X_test)
    print("Selected Features:",i)
    runRandomForestClassifier(X_train_rfe_model,X_test_rfe_model,y_train,y_test)
    print("**************************")

Selected Features: 1
Accuracy Score: 0.8168
Precision Score: 0.6863
Recall Score 0.3165
Confusion Matrix:
 [[4481  192]
 [ 907  420]]
**************************
Selected Features: 2
Accuracy Score: 0.8182
Precision Score: 0.6941
Recall Score 0.318
Confusion Matrix:
 [[4487  186]
 [ 905  422]]
**************************
Selected Features: 3
Accuracy Score: 0.818
Precision Score: 0.6911
Recall Score 0.3203
Confusion Matrix:
 [[4483  190]
 [ 902  425]]
**************************
Selected Features: 4
Accuracy Score: 0.819
Precision Score: 0.691
Recall Score 0.3286
Confusion Matrix:
 [[4478  195]
 [ 891  436]]
**************************
Selected Features: 5
Accuracy Score: 0.8187
Precision Score: 0.6912
Recall Score 0.3255
Confusion Matrix:
 [[4480  193]
 [ 895  432]]
**************************
Selected Features: 6
Accuracy Score: 0.818
Precision Score: 0.6799
Recall Score 0.3346
Confusion Matrix:
 [[4464  209]
 [ 883  444]]
**************************
Selected Features: 7
Accuracy Score: 0.

## We can see the above evaluation looks like same for all the feature

# THE END !!!